# Phase 1b - Preprocessing plan for the machine-learning models (CICIDS2017)

Starts from the **raw CSV files** so every decision is explicit and reproducible. Output: one clean, de-duplicated,
leakage-checked train/val/test split in two views.

| Step | What | Why (from your results) |
|---|---|---|
| 1 | Load, strip headers, fix label text, drop inf/NaN rows, drop duplicate column | Standard CICIDS2017 clean-up |
| 2 | Binary label + keep `attack_type` and `day` | Per-attack analysis and an optional day-based test later |
| 3 | Drop exact duplicate rows and rows whose identical features have conflicting labels | Duplicates leak between splits (XGBoost AUC of 1.0000 is a warning sign) |
| 4 | Stratified 70/15/15 split on attack type | Rare attacks appear in every split |
| 5 | Drop constant and near-duplicate features (train only) | Redundant features make adversarial perturbations unrealistic |
| 6 | Shortcut-feature ablation | `Bwd Packet Length Min` and `Destination Port` dominated the trees |
| 7 | Build two views: raw (trees) and signed-log + StandardScaler (LR, SVM, K-Means, deep models) | Min-Max scaling is why LR/SVM/K-Means were poor |
| 8 | Remove val/test rows that become duplicates of train after feature drops, then verify | Final leakage guard |

No SMOTE and no class weights. Fix precision by tuning the decision threshold on validation.

In [3]:
import os, glob, json, joblib
import numpy as np, pandas as pd
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler
from sklearn.metrics import confusion_matrix, roc_auc_score, roc_curve, precision_recall_curve

# ===================== CONFIG (edit here) =====================
DATA_DIR  = "data/processed"   # folder containing the 8 CICIDS2017 CSV files
OUT_DIR   = "ml_prep"              # all outputs go here
SEED      = 42
VAL_FRAC  = 0.15
TEST_FRAC = 0.15
CORR_THR  = 0.99                   # drop one of each feature pair correlated above this
# ==============================================================
os.makedirs(OUT_DIR, exist_ok=True)

def slog1p(a):
    """Signed log: keeps negative values (e.g. -1 sentinels) instead of clipping them to 0."""
    a = np.asarray(a, dtype="float64")
    return np.sign(a) * np.log1p(np.abs(a))

## Step 1 - Load and clean

In [2]:
files = sorted(glob.glob(os.path.join(DATA_DIR, "*.csv")))
assert files, f"No CSV files found in {DATA_DIR}"
parts = []
for f in files:
    d = pd.read_csv(f, skipinitialspace=True, low_memory=False, encoding="latin-1")
    d.columns = d.columns.str.strip()
    feats = [c for c in d.columns if c != "Label"]
    d[feats] = d[feats].apply(pd.to_numeric, errors="coerce").astype("float32")
    d["day"] = os.path.basename(f).split("-")[0]
    parts.append(d)
    print(f"{os.path.basename(f):55s} {len(d):>10,} rows")
df = pd.concat(parts, ignore_index=True)
del parts

# the web-attack labels contain a broken dash character in the original files
df["Label"] = (df["Label"].astype(str)
               .str.replace(r"[^\x00-\x7F]+", "-", regex=True)
               .str.replace(r"\s*-\s*", " - ", regex=True)
               .str.strip())

FEATS = [c for c in df.columns if c not in ("Label", "day")]
if "Fwd Header Length.1" in FEATS and df["Fwd Header Length.1"].equals(df["Fwd Header Length"]):
    df = df.drop(columns="Fwd Header Length.1")
    FEATS.remove("Fwd Header Length.1")
    print("dropped duplicate column 'Fwd Header Length.1'")

n0 = len(df)
df[FEATS] = df[FEATS].replace([np.inf, -np.inf], np.nan)
df = df.dropna(subset=FEATS).reset_index(drop=True)
print(f"rows: {n0:,} -> {len(df):,}  (removed {n0 - len(df):,} with inf/NaN)  | features: {len(FEATS)}")

AssertionError: No CSV files found in MachineLearningCVE

## Step 2 - Labels

In [ ]:
df["attack_type"] = df["Label"]
df["y"] = (df["Label"] != "BENIGN").astype("int8")
df = df.drop(columns="Label")
print(df["attack_type"].value_counts().to_string())
print("\nattack share:", round(float(df["y"].mean()), 4))

## Step 3 - Remove duplicates and label conflicts

Identical feature vectors with different labels cannot be learned and are removed entirely.
Exact duplicates are reduced to one copy so the same flow can never sit in both train and test.

In [ ]:
df["_h"] = pd.util.hash_pandas_object(df[FEATS], index=False).to_numpy()
n0 = len(df)
n_labels = df.groupby("_h")["y"].transform("nunique")
n_conflict = int((n_labels > 1).sum())
df = df[n_labels == 1]
n1 = len(df)
df = df.drop_duplicates(subset="_h").reset_index(drop=True)
print(f"start {n0:,} | conflicting-label rows removed {n_conflict:,} | exact duplicates removed {n1 - len(df):,} | kept {len(df):,}")
print("attack share after cleaning:", round(float(df["y"].mean()), 4))
print(df["attack_type"].value_counts().to_string())

## Step 4 - Stratified split (train / validation / test)

Stratified on attack type, so even very rare attacks appear in all three splits. Classes with fewer than 20 rows
are pooled for stratification only. The test set is untouched until final evaluation.

In [ ]:
key = df["attack_type"].copy()
vc = key.value_counts()
key[key.isin(vc[vc < 20].index)] = "RARE"
if (key == "RARE").sum() < 10:
    key[key == "RARE"] = "BENIGN"

tr, tmp = train_test_split(df.index.to_numpy(), test_size=VAL_FRAC + TEST_FRAC,
                           stratify=key, random_state=SEED)
va, te = train_test_split(tmp, test_size=TEST_FRAC / (VAL_FRAC + TEST_FRAC),
                          stratify=key.loc[tmp], random_state=SEED)

split = pd.Series("train", index=df.index, name="split")
split.loc[va] = "val"
split.loc[te] = "test"
print(pd.crosstab(df["attack_type"], split)[["train", "val", "test"]].to_string())
print("\nattack share train/val/test:", [round(float(df.loc[i, "y"].mean()), 4) for i in (tr, va, te)])

## Step 5 - Feature pruning (decided on the training split only)

Constant columns carry no information. Features that are almost perfectly correlated (for example total bytes
and subflow bytes) are mathematically tied together: in the adversarial phase, changing one without the other would
create impossible traffic, so we keep one of each pair.

In [ ]:
Xtr_raw = df.loc[tr, FEATS]
const = [c for c in FEATS if Xtr_raw[c].nunique() <= 1]
feats1 = [c for c in FEATS if c not in const]

samp = pd.DataFrame(slog1p(Xtr_raw[feats1].sample(min(300_000, len(Xtr_raw)), random_state=SEED)),
                    columns=feats1)
corr = samp.corr().abs()
upper = corr.where(np.triu(np.ones(corr.shape, dtype=bool), k=1))
redundant = [c for c in upper.columns if (upper[c] > CORR_THR).any()]
feats2 = [c for c in feats1 if c not in redundant]

print("constant features :", const)
print("redundant features:")
for c in redundant:
    print(f"   {c:35s} ~ {upper[c].idxmax()}  (|r|={upper[c].max():.4f})")
print(f"\nfeatures: {len(FEATS)} -> {len(feats2)}")
del Xtr_raw, samp, corr, upper

## Step 6 - Shortcut-feature ablation

Quick XGBoost on a 600k sample of train, scored on validation. If dropping a feature barely changes the result,
the model does not need it and the baseline is more credible without it. If scores collapse without it, the dataset
is leaning on an artifact - say so in your report.

In [ ]:
def metrics(y, p, thr):
    y = np.asarray(y).astype(int)
    pred = (p >= thr).astype(int)
    tn, fp, fn, tp = confusion_matrix(y, pred, labels=[0, 1]).ravel()
    return dict(threshold=float(thr),
                accuracy=float((tp + tn) / len(y)),
                precision=float(tp / max(tp + fp, 1)),
                recall=float(tp / max(tp + fn, 1)),
                f1=float(2 * tp / max(2 * tp + fp + fn, 1)),
                fpr=float(fp / max(fp + tn, 1)),
                roc_auc=float(roc_auc_score(y, p)),
                tn=int(tn), fp=int(fp), fn=int(fn), tp=int(tp))

def thr_f1(y, p):
    pr, rc, th = precision_recall_curve(np.asarray(y).astype(int), p)
    f1 = 2 * pr * rc / (pr + rc + 1e-12)
    return float(th[np.argmax(f1[:-1])])

def thr_fpr(y, p, max_fpr=0.01):
    fpr, tpr, th = roc_curve(np.asarray(y).astype(int), p)
    return float(th[np.where(fpr <= max_fpr)[0][-1]])

In [ ]:
import xgboost as xgb

ytr_all, yva_all = df.loc[tr, "y"].to_numpy(), df.loc[va, "y"].to_numpy()
sub = np.random.default_rng(SEED).choice(len(tr), size=min(600_000, len(tr)), replace=False)
Xa, ya = df.loc[tr, feats2].iloc[sub], ytr_all[sub]
Xv = df.loc[va, feats2]

rows = []
for drop in ([], ["Destination Port"], ["Bwd Packet Length Min"], ["Destination Port", "Bwd Packet Length Min"]):
    drop = [c for c in drop if c in feats2]
    cols = [c for c in feats2 if c not in drop]
    m = xgb.XGBClassifier(n_estimators=200, max_depth=6, learning_rate=0.1,
                          tree_method="hist", n_jobs=-1, random_state=SEED)
    m.fit(Xa[cols], ya)
    r = metrics(yva_all, m.predict_proba(Xv[cols])[:, 1], 0.5)
    rows.append(dict(dropped=", ".join(drop) or "(none)", f1=r["f1"], precision=r["precision"],
                     recall=r["recall"], fpr=r["fpr"], auc=r["roc_auc"]))
pd.DataFrame(rows).round(4)

## Step 7 - Final feature set, final leakage guard, and saving both views

Recommended default: drop `Destination Port`. In this lab dataset specific attacks use specific ports, so the model can
learn the port instead of the behaviour, and an attacker cannot change the port without changing the attack.
Edit `DROP_FEATURES` using the ablation table above.

- **Raw view** (`X_*_raw.parquet`): cleaned but unscaled. Use for XGBoost and Decision Tree.
- **Scaled view** (`X_*_scaled.npy`): signed-log, then `StandardScaler` fitted on train, then clipped to +/-5.
  Use for Logistic Regression, SVM, K-Means and the deep models.

In [ ]:
DROP_FEATURES = ["Destination Port"]          # <- edit after reading the ablation table
FINAL = [c for c in feats2 if c not in DROP_FEATURES]

# dropping features can create NEW duplicates across splits: remove them from val/test
hh = pd.util.hash_pandas_object(df[FINAL], index=False).to_numpy()
tr_h = hh[tr]
n_va, n_te = len(va), len(te)
va = va[~np.isin(hh[va], tr_h)]
te = te[~np.isin(hh[te], tr_h)]
te = te[~np.isin(hh[te], hh[va])]
print(f"val rows removed (duplicate of train): {n_va - len(va):,} | test rows removed: {n_te - len(te):,}")

scaler = StandardScaler().fit(slog1p(df.loc[tr, FINAL]))      # TRAIN ONLY

summary = {"features": FINAL, "dropped_constant": const, "dropped_redundant": redundant,
           "dropped_by_choice": DROP_FEATURES, "splits": {}}
for name, idx in (("train", tr), ("val", va), ("test", te)):
    X = df.loc[idx, FINAL].reset_index(drop=True)
    y = df.loc[idx, "y"].to_numpy()
    meta = df.loc[idx, ["attack_type", "day"]].reset_index(drop=True).assign(y=y)
    X.to_parquet(f"{OUT_DIR}/X_{name}_raw.parquet")
    meta.to_parquet(f"{OUT_DIR}/meta_{name}.parquet")
    np.save(f"{OUT_DIR}/y_{name}.npy", y)
    Z = np.clip(scaler.transform(slog1p(X)), -5, 5).astype("float32")
    np.save(f"{OUT_DIR}/X_{name}_scaled.npy", Z)
    summary["splits"][name] = {"rows": int(len(y)), "attack_share": float(y.mean())}
    print(f"{name:5s} rows={len(y):>9,}  attack share={y.mean():.4f}  scaled range=({Z.min():.2f}, {Z.max():.2f})")

joblib.dump({"scaler": scaler, "columns": FINAL, "clip": 5.0, "transform": "signed_log1p"},
            f"{OUT_DIR}/scaler_slog_std.joblib")
json.dump(summary, open(f"{OUT_DIR}/preprocessing_report.json", "w"), indent=2)
print("\nsaved to", OUT_DIR, "| final features:", len(FINAL))

## Step 8 - Verify, then sanity-check with two models

The checks below must pass before any model is trained. The mini-benchmark confirms the plan worked:
XGBoost on the raw view, Logistic Regression on the scaled view, and a benign-only K-Means anomaly detector.
Thresholds are chosen on validation and applied to test. Your old LR/SVM numbers (precision about 45%) should improve sharply.

In [ ]:
Xr = {n: pd.read_parquet(f"{OUT_DIR}/X_{n}_raw.parquet") for n in ("train", "val", "test")}
Zs = {n: np.load(f"{OUT_DIR}/X_{n}_scaled.npy") for n in ("train", "val", "test")}
Y  = {n: np.load(f"{OUT_DIR}/y_{n}.npy") for n in ("train", "val", "test")}

H = {n: set(pd.util.hash_pandas_object(Xr[n], index=False).to_numpy()) for n in Xr}
assert not (H["train"] & H["val"]),  "train/val overlap"
assert not (H["train"] & H["test"]), "train/test overlap"
assert not (H["val"] & H["test"]),   "val/test overlap"
for n in Xr:
    assert np.isfinite(Xr[n].to_numpy()).all() and np.isfinite(Zs[n]).all(), f"non-finite values in {n}"
    assert Xr[n].shape[0] == len(Y[n]) == Zs[n].shape[0]
assert list(Xr["train"].columns) == list(Xr["val"].columns) == list(Xr["test"].columns)
print("All checks passed: no cross-split duplicates, all values finite, columns aligned.")

In [ ]:
from sklearn.linear_model import LogisticRegression
from sklearn.cluster import MiniBatchKMeans

def three_rows(name, pva, pte):
    out = []
    for rule, t in (("thr_0.5", 0.5), ("thr_best_f1", thr_f1(Y["val"], pva)),
                    ("thr_fpr<=1%", thr_fpr(Y["val"], pva, 0.01))):
        m = metrics(Y["test"], pte, t)
        out.append(dict(model=name, rule=rule, thr=m["threshold"], acc=m["accuracy"], precision=m["precision"],
                        recall=m["recall"], f1=m["f1"], fpr=m["fpr"], auc=m["roc_auc"],
                        false_alarms=m["fp"], missed=m["fn"]))
    return out

res = []

xg = xgb.XGBClassifier(n_estimators=100, max_depth=6, learning_rate=0.3, tree_method="hist",
                       n_jobs=-1, random_state=SEED).fit(Xr["train"], Y["train"])
res += three_rows("XGBoost (raw)", xg.predict_proba(Xr["val"])[:, 1], xg.predict_proba(Xr["test"])[:, 1])

lr = LogisticRegression(max_iter=300).fit(Zs["train"], Y["train"])        # no class_weight
res += three_rows("LogReg (scaled)", lr.predict_proba(Zs["val"])[:, 1], lr.predict_proba(Zs["test"])[:, 1])

km = MiniBatchKMeans(n_clusters=20, random_state=SEED, batch_size=10_000, n_init=3).fit(Zs["train"][Y["train"] == 0])
dva, dte = km.transform(Zs["val"]).min(axis=1), km.transform(Zs["test"]).min(axis=1)   # distance to nearest benign centroid
res += three_rows("K-Means benign-only (scaled)", dva, dte)

pd.DataFrame(res).round(4)

## What to do with the output

- **XGBoost drops a little, to roughly 99.5% accuracy:** expected and more defensible. Report it.
- **Logistic Regression precision rises well above 45%:** the scaling was the problem. Now retrain SVM
  (use `LinearSVC` or `SGDClassifier` on the scaled view) and K-Means on these files.
- **Anything still poor after threshold tuning:** that model is simply weaker on this data. Say so.
- Retrain the deep models on `X_*_scaled.npy` only after you are happy with this split. In
  `phase2b_dl_retrain.ipynb`, load these arrays instead of running its own scaling step.